# Train one YOLO model on Colab

Trains a single model variant end-to-end and persists the run folder to Google Drive.
Run one model per Colab session (free tier ~12h is not enough for all variants at once).

**One-time setup:** in Colab, click the 🔑 key icon (left sidebar) → add a secret
named `GITHUB_TOKEN` with a fine-grained PAT that has read access to this repo,
and toggle 'Notebook access' on.

Before running: upload `dataset.zip` + `dataset.meta.json` to
`MyDrive/yolo-pipeline/datasets/v1/`.

In [ ]:
# === EDIT THESE PER SESSION ===
REPO_URL    = "https://github.com/tahmid013/yolo.git"
REPO_BRANCH = "main"
DATASET_VERSION = "v1"
MODEL       = "yolo11s"                  # one of: yolo11n yolo11s yolo11m yolo11l yolo12s
CONFIG      = "configs/yolo11s.yaml"     # matching variant config
RESUME      = False                      # set True to continue the latest Drive run for MODEL
# ==============================

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os, shutil, subprocess
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Set the GITHUB_TOKEN secret in the 🔑 Secrets panel (left sidebar).')

if os.path.isdir('/content/code'):
    shutil.rmtree('/content/code')

url = REPO_URL.replace('https://', f'https://{token}@')
subprocess.run(
    ['git', 'clone', '--quiet', '--branch', REPO_BRANCH, url, '/content/code'],
    check=True,
)
os.chdir('/content/code')
print('cloned at:', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())

In [ ]:
!pip install -q -r requirements.txt
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
from pipeline.dataset import ensure_dataset
from pipeline import paths

data_yaml = ensure_dataset(
    zip_path=paths.dataset_zip(DATASET_VERSION),
    meta_path=paths.dataset_meta(DATASET_VERSION),
    target=paths.LOCAL_DATASET,
)
print('data.yaml:', data_yaml)

In [ ]:
from pipeline.persist import cleanup_tmp
cleanup_tmp(paths.RUNS_DIR)

In [ ]:
from pipeline.train import run as train_run
run_dir = train_run(
    model=MODEL,
    config=CONFIG,
    drive_runs_dir=paths.RUNS_DIR,
    local_runs_dir=paths.LOCAL_RUNS,
    data_yaml=data_yaml,
    dataset_meta_path=paths.dataset_meta(DATASET_VERSION),
    base_config='configs/base.yaml',
    resume=RESUME,
)
print('Drive run dir:', run_dir)

In [ ]:
from pipeline.evaluate import run as eval_run
eval_payload = eval_run(
    run_dir=run_dir,
    data_yaml=data_yaml,
    drive_runs_dir=paths.RUNS_DIR,
)
print('test mAP50:', eval_payload['overall']['mAP50'])
print('test mAP50-95:', eval_payload['overall']['mAP50_95'])